day07

0. 복습
랜덤 포레스트
 : 조금씩 다르게 학습한 결정 나무 여러 그루의 예측을 모아
   다수결로 결정하는 모델

1) 앙상블
- 여러 모델을 모아 더 좋은 성능을 내는 방법 

2) 배깅
- 랜덤포레스트는 각 결정 나무에 조금씩 다른 데이터를 주고
  무작위로 일부를 뽑아 다른 훈련 데이터를 준다

+) GridSearchCV
 : 하이퍼파라미터 후보 값들만 작성하면
   모든 조합을 대신 시도해 가장 좋은 조합을 알려주는 도구
 
1. 그래디언트 부스팅(Gradient Boosting)
 : 앞의 모델이 틀린 오차를 다음 모델이 이어서 보완하도록,
   나무를 순서대로 쌓아 성능을 끌어올리는 앙상블
- 랜덤포레스트가 "여러 전문가의 동시 다수결"이라면,
  부스팅은 "한명씩 이어달리며 앞 사람의 실수를 메우는 팀"
- 나무 한 그루는 약하지만, 약한 나무를 수백 그루 이어붙이며
  조금씩 보정하면 강력한 모델이 된다(각 나무를 약한 학습기라 부른다)

1) 배깅 vs 부스팅
 : 둘다 "나무를 여러 그루 모으는" 앙상블이지만,
   나무를 모으는 방식이 다르다
		배깅(랜덤포레스트)		부스팅
=====================================================================
나무를 키우는 순서 동시에(서로 독립)		한 그루씩 순서대로
나무끼리 관계 	   무관				앞 나무의 오차를 보완
합치는 법	   다수결 투표			순서대로 더해가며 보정	
특정		   과적합을 분산으로 낮춤	오차를 집요하게 줄임
		   여러 명이 동시에 풀고 다수결	이어달리며 앞의 실수를 보완

** 배깅은 나무들이 제각각 독립적으로 학습해 마지막에 투표한다
   부스팅은 나무들이 줄줄이 이어지며 앞 나무의 오차를 다음 나무가 메운다	
2) 어떻게 오차를 줄이는지(학습률)
 : 부스팅의 각 나무는 앞까지의 예측이 틀린 만큼(오차)을 목표로 삼아
   그 오차를 줄이는 방향으로 조금씩 보정
- 이때 한번에 얼마나 보정할지를 정하는 값이 학습률(learning_rate)
	* 학습률이 작으면 : 나무가 아주 조금씩만 고친다
			    그만큼 나무가 많이 필요하다
	* 학습률이 크면 : 나무가 많이 고친다
			  빠르지만 학습 데이터에 과하게 맞춰(과적합)
			  버리기 쉽다
- 그래서 부스팅에서는 학습률(learning_rate)과 나무 수(n_estimator)
  를 짝으로 생각한다
  (학습률은 작게, 나무는 많이 주는 것이 안정적이다)

3) 하이퍼파라미터
- 부스팅에서 가장 중요한 하이퍼파라미터는 learning_rate(학습률)
  n_estimators(나무 수)이고, 둘은 짝으로 움직인다
(1) learning_rate : 한 나무가 고치는 정도
		- 크게하면 : 빠르지만 과적합 위험
 		- 작게하면 : 나무가 많이 필요
(2) n_estimators : 나무(단계) 수
		- 크게하면 : 더 정교하나 느림, 과적합
		- 작게하면 : 덜 정교(과소적합)
(3) max_depth : 각 나무의 깊이
		(보통 얕게(2 ~ 4)쓴다) 

4) XGBoost, LightGBM
- GradientBoostingClassifier는 학습에는 좋지만,
  실무, 대회에서는 더 빠르고 강력한 XGBoost, LightGBM을 사용한다

5) 정리
- 부스팅 = 나무를 한그루씩 순서대로 쌓되,
    	   앞의 오차를 다음 나무가 보정하는 앙상블
- 강점 : 학습 데이터를 덜 외우면서 성능이 높다(과적합에 강함)
- 하이퍼파라미터 : learning_rate(학습률)와 n_estimators(나무 수)
	보통 "학습률 작게 + 나무 많이"

## 그래디언트 부스팅
> 타이타닉 승객들의 생존 여부를 분류하는 부스팅 모델

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

# 타이타닉 데이터
titanic = sns.load_dataset('titanic')
df = titanic[['survived', 'pclass', 'sex', 'age', 'fare']]

print(df.info())

df.head()

In [ ]:
# 전처리
# 1) 성별 : 문자열 -> 숫자(남=0, 여=1)
df['sex'] = df['sex'].map({"male" : 0, "female":1})
# 2) 나이열의 결측치를 나이의 중앙값으로 대체
df['age'] = df['age'].fillna(df['age'].median())

# 특성(독립변수)/타깃(종속변수)
X = df.drop(columns= 'survived') # survived 열을 제외한 열의 특성
y = df['survived']

# 훈련/테스트 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### 학습, 예측, 평가

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier # 부스팅 모델
from sklearn.metrics import accuracy_score, classification_report

# 부스팅 모델 생성 후 학습(기본 learning_rate=0.1, n_estimators=100)
boosting = GradientBoostingClassifier(random_state=42)
boosting.fit(X_train, y_train)

# 테스트 데이터로 예측 후 평가
pred = boosting.predict(X_test)
print(f"정확도 : {accuracy_score(y_test, pred) : .3f}")
print(classification_report(y_test, pred, target_names=['사망(0)', "생존(1)"]))

# 기본 설정만으로는 정확도가 0.81 결정나무와 랜덤 포레스트보다 조금 높다
# (거의 차이 안남)
# 진짜 차이는 과적합 비교에서 드러난다

### 나무 하나 vs 숲 vs 부스팅
> 3개의 모델을 만들어  학습용, 평가용 정확도를 함께 보면 과적합 정도를 알 수 있음

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 3개의 모델을 딕셔너리에 담아 한번에 비교 
models = {
    "결정나무" : DecisionTreeClassifier(random_state=42),
    "랜덤포레스트" : RandomForestClassifier(random_state=42),
    "부스팅" : GradientBoostingClassifier(random_state=42)
}

# 3개의 모델 학습, 예측, 평가(학습용, 평가용)
for name, model in models.items() :
    model.fit(X_train, y_train)
    # 학습용, 평가용에 대한 정확도
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name} | 학습용 {train_acc:.3f} | 평가용 {test_acc:.3f}")

# 결정나무는 학습용과 평가용 사이 간격이 넓다 => 크게 과적합
# 랜덤포레스트는 투표로 과적합을 눌러 평가용 0.793 개선
# 부스팅은 학습용이 0.897로 오히려 낮다 => 학습 데이터를 통째로 외우기보다
# 오차를 차근차근 보정하다 보니, 학습용과 평가용의 간격이 가장 좁다
# => 과적합이 가장 적고 평가용 성능은 가장 높다

# ** 부스팅은 "학습 데이터를 덜 외우면서 새 데이터는 더 잘 맞히는" 균형을 보여준다(부스팅의 장점)

## 최적의 학습률

In [ ]:
# 학습률을 0.01 ~ 1.0까지 바꿔가며 학습용, 평가용 정확도 비교 
for lr in [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]:
    model = GradientBoostingClassifier(learning_rate=lr,
                                      n_estimators=100,
                                      random_state=42)
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"학습률={lr} | 학습용 {train_acc:.3f} | 평가용 {test_acc:.3f}")

# 학습용 정확도는 학습률이 커질수록 계속 오른다 => 
# 학습 데이터에 점점 더 맞춰간다는 뜻
# 평가용은 중간(0.3 부근)에서 정점을 찍고 다시 내려간다. =>
# 학습률이 너무 크면 학습 데이터에 과하게 맞춰 과적합되기 때문이다

# ** 학습률이 너무 작아도(과소적합), 너무 커도(과적합) 안된다

### 최적의 하이퍼파라미터 적용(튜닝)
> GridSearchCV로 최적 조합을 찾기(학습률, 나무 수, 깊이)

In [ ]:
from sklearn.model_selection import GridSearchCV

# 3개의 하이퍼파라미터 조합을 교차검증으로 탐색
param_grid = {
    "learning_rate" : [0.01, 0.05, 0.1, 0.2],
    "n_estimators" : [100, 200, 300],
    "max_depth" : [2, 3, 4]
}

grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid, cv=5, n_jobs=-1
)
grid.fit(X_train, y_train)

# 최적의 조합
print(f"최적 조합 : {grid.best_params_}")

# 기본 모델 vs 튜닝 모델
base = GradientBoostingClassifier(random_state=42).fit(X_train, y_train)
print(f"기본 모델 정확도 : {accuracy_score(y_test, base.predict(X_test)) : .3f}")
print(f"튜닝 모델 정확도 : {accuracy_score(y_test, grid.best_estimator_.predict(X_test)) : .3f}")

# 튜닝으로 정확도가 0.827로 오름
# 최적 조합을 보면 학습률은 작게(0.1), 대신 나무는 많이(300), 깊이는 얕게(2)

### 특성 중요도

In [ ]:
importance = pd.DataFrame({
    "특성" : ['객실등급', '성별', '나이', '요금'],
    "중요도" : boosting.feature_importances_.round(3)
}).sort_values("중요도", ascending=False)
importance

### +) 실무도구

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# XGBoost
xgb = XGBClassifier(random_state=42, eval_metric="logloss")
xgb.fit(X_train, y_train)
print(f"XGBoost 정확도 : {accuracy_score(y_test, xgb.predict(X_test)) : .3f}")

# LightGBM
lgbm = LGBMClassifier(random_state=42, verbose=-1)
lgbm.fit(X_train, y_train)
print(f"LightGBM 정확도 : {accuracy_score(y_test, lgbm.predict(X_test)) : .3f}")

# 데이터가 크고 복잡할수록 위의 모델이 훨씬 빠르고 성능도 좋은 경우가 많다

In [ ]:
## <부스팅 모델 실습>
from sklearn.datasets import load_breast_cancer
# 유방암 진단 데이터
# 종양 측정값으로 양성/악성 분류
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns= data.feature_names) # 특성 30개
y = data.target # 0 =악성, 1 = 양성
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# 1) 부스팅 모델 하이퍼파라미터 기본값으로 학습하고 test 정확도
base = GradientBoostingClassifier(random_state=42).fit(X_train, y_train)
print(f"기본 모델 정확도 : {accuracy_score(y_test, base.predict(X_test)) : .3f}")

In [ ]:
# 2) 최적의 하이퍼파라미터 찾기(GridSearchCV)
# 학습률(0.01, 0.05, 0.1, 0.2)와 나무 개수 (100, 200, 300)를 튜닝해
# best_params_ 출력

param_grid = {
    "learning_rate" : [0.01, 0.05, 0.1, 0.2],
    "n_estimators" : [100, 200, 300]
}

grid = GridSearchCV(GradientBoostingClassifier(random_state=42),
                   param_grid, cv=5, n_jobs=-1)
grid.fit(X_train, y_train)
print(f"최적 조합 : {grid.best_params_}")
print(f"최적 교차검증 점수 : {grid.best_score_ : .3f}")
print(f"튜닝 모델 정확도 : {accuracy_score(y_test, grid.best_estimator_.predict(X_test)) : .3f}")
# 기본 정확도 0.956, 튜닝 후에도 0.956이 나온다
# => 최적 조합이 사실상 기본값

In [ ]:
# 3) 튜닝 모델의 classification_report 확인(의료 진단이라 재현율 중요)
pred = grid.best_estimator_.predict(X_test)
print(classification_report(y_test, pred, target_names=["악성(0)", "양성(1)"]))
# 재현율 : 악성(0)의 재현율 0.93
# - 실제 악성 43건 중 93%를 잡아냈다
# - 의료진단에서는 악성을 놓치지 않는 것(재현율)이 중요하므로
# - 필요하면 임계값을 조절해 더 높일 수 있다

### 과제

# 그래디언트 부스팅 (Gradient Boosting) 과제

그래디언트 부스팅은 나무를 **한 그루씩 순서대로 쌓되, 앞 나무가 틀린 부분을 다음 나무가 보완**하는 앙상블이다.
랜덤 포레스트가 "여러 그루를 동시에 키워 다수결"이었다면, 이번엔 **"이어달리며 앞의 실수를 메우기"** 다.
이번 과제는 ① 세 모델을 나란히 세워 **부스팅의 성격**을 확인하고(문제 1), ② 핵심 손잡이인
**학습률(`learning_rate`)** 과 튜닝을 다룬다(문제 2).

- 공통 데이터: **`day07_고객이탈.csv`**(통신사 고객 800명)

## 문제 1) 세 모델 대결 — 결정나무 vs 랜덤 포레스트 vs 부스팅

수업에서 **나무 한 그루(0.675)보다 숲(0.800)이 낫다**는 것을 확인했다. 그렇다면 부스팅은 어떨까?
같은 데이터에 **세 모델을 나란히 세워** 비교해보자. 정확도 하나만 보지 말고 **학습용과 평가용을 함께** 보는 것이 핵심이다.

1. `day07_고객이탈.csv`를 불러와 `X`(5개 특성)와 `y`(`이탈`)로 나누고 **8:2 분할**(`random_state=42`)한다.
2. 세 모델을 **모두 기본 설정으로** 학습하고, 각각의 **학습용·평가용 정확도**를 출력한다.
   - `DecisionTreeClassifier` / `RandomForestClassifier(n_estimators=100)` / `GradientBoostingClassifier`
   - 힌트: 세 모델을 **딕셔너리에 담아 반복문**으로 돌리면 코드가 짧아진다.
3. 부스팅의 **`classification_report`** 를 출력한다.
4. **나이 35 · 월요금 90000 · 통화시간 150 · 문의횟수 4 · 약정개월 3** 인 고객을 예측한다. (랜덤 포레스트와 같은 답이 나오는가?)
5. 부스팅의 **특성 중요도**를 표로 출력하고, day07 숲의 중요도(`월요금` 0.242 > `약정개월` 0.210 > `문의횟수` 0.193 > `통화시간` 0.185 > `나이` 0.169)와 **비교**한다.
6. 결과를 **해석**한다. (부스팅만 학습용 정확도가 낮은 이유는 / 어느 모델이 과적합이 가장 심한가 / 숲과 부스팅의 중요도는 어떻게 다른가)

In [ ]:
# day07_고객이탈.csv : 통신사 고객의 이용 기록과 해지 여부 (가상 데이터 800건)
#   - 나이     : 고객 나이 (20~69세)
#   - 월요금   : 한 달 요금 (원, 2만~12만)
#   - 통화시간 : 한 달 통화시간 (분, 10~800)
#   - 문의횟수 : 고객센터에 문의한 횟수 (0~6회)
#   - 약정개월 : 남은 약정 기간 (개월, 1~48)
#   - 이탈     : 해지 여부 (1=이탈, 0=유지)  ← 우리가 맞힐 정답(타깃)
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('./day07_고객이탈.csv')
# df.head()

X = df[['나이', '월요금', '통화시간', '문의횟수', '약정개월']]
y = df['이탈']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    'tree' : DecisionTreeClassifier(),
    'forest' : RandomForestClassifier(n_estimators=100),
    'boost' : GradientBoostingClassifier()    
}

for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    print(name)
    print(f"학습용 : {accuracy_score(y_train, train_pred)}")
    print(f"평가용 : {accuracy_score(y_test, test_pred)}")

boost_pred = models['boost'].predict(X_test)
print(classification_report(y_test, boost_pred, target_names=["유지(0)", '이탈(1)']))

customer = pd.DataFrame( 
    [[35, 90000, 150, 4, 3]],
    columns=X.columns
)

boost_result = models['boost'].predict(customer)
forest_result = models['forest'].predict(customer)
print(boost_result)
print(forest_result)

importance = pd.DataFrame({
    "특성" : ['나이', '월요금', '통화시간', '문의횟수', '약정개월'],
    "중요도" : models['boost'].feature_importances_.round(3)
}).sort_values('중요도', ascending=False)

importance

# 부스팅이 학습용이 낮은 이유는 나무를 순서대로 만들면서 앞 모델을 보완하는 방식이다. 그렇다고 해서 무조건 낮다고 나쁜건 아니다
# 과적합이 가장 심한 것은 tree이다. 학습 데이터를 너무 세세하게 외웠다.
# 포레스트는 월요금, 부스팅은 문의횟수가 중요하다

## 문제 2) 학습률(`learning_rate`) — 부스팅의 핵심 손잡이

부스팅에는 랜덤 포레스트에 없던 손잡이, **학습률**이 있다. 한 그루가 앞의 오차를 **얼마나 세게 고칠지**를 정하는 값이다.
너무 작으면 굼뜨고, 너무 크면 학습 데이터에 과하게 맞춰버린다. 직접 돌려 확인하고, 마지막엔 **튜닝으로 최적 조합**까지 찾아보자.

1. 같은 `day07_고객이탈.csv`를 **8:2 분할**(`random_state=42`)한다.
2. **`learning_rate`를 0.01 · 0.05 · 0.1 · 0.3 · 0.5 · 1.0** 으로 바꿔가며 학습하고,
   각 경우의 **학습용·평가용 정확도**를 출력한다. (반복문)
3. **`GridSearchCV`** 로 `learning_rate` · `n_estimators` · `max_depth`의 최적 조합을 찾고(`cv=5`),
   **`best_params_`·`best_score_`** 와 **기본 모델 대비 test 정확도**를 비교한다. (11회차 복습)
4. (도전) 같은 부스팅 계열의 실무 도구 **XGBoost·LightGBM**으로도 학습해 정확도를 비교한다.

5. 결과를 **해석**한다. (학습률이 커지면 학습용/평가용은 각각 어떻게 되나 / 과적합은 어디서부터 /
   적당한 학습률은 / 튜닝이 찾은 조합이 2번 결과와 맞아떨어지는가)

In [ ]:
from sklearn.model_selection import GridSearchCV

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)
learning_rate = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]
for rate in learning_rate:
    model = GradientBoostingClassifier(learning_rate=rate)
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    print(rate)
    print(f"학습용 : {accuracy_score(y_train, train_pred)}")
    print(f"평가용 : {accuracy_score(y_test, test_pred)}")

gb = GradientBoostingClassifier(random_state=42)
params = {
    "learning_rate" : [0.01, 0.05, 0.1, 0.3, 0.5, 1.0],
    "n_estimators" : [100, 200, 300],
    'max_depth' : [1, 2, 3]
}
grid = GridSearchCV(
    gb, params, cv=5, n_jobs= -1 
)
grid.fit(X_train, y_train)
print()
print(f"최적 조합 : {grid.best_params_}")
print(f"최고 교차검증 점수 : {grid.best_score_}")
print(f"최적 모델 test 정확도 : {accuracy_score(y_test, grid.best_estimator_.predict(X_test))}")
print(f"기본 모델 test 정확도 : {accuracy_score(y_test, models['boost'].predict(X_test))}")

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

xgb = XGBClassifier(random_state=42, eval_metric="logloss")
xgb.fit(X_train, y_train)
print(f"XGB 정확도 : {accuracy_score(y_test, xgb.predict(X_test)) : .3f}")

lgbm = LGBMClassifier(random_state=42, verbose=-1)
lgbm.fit(X_train, y_train)
print(f"LGBM 정확도 : {accuracy_score(y_test, lgbm.predict(X_test)) : .3f}")

# 학습률이 커질수록 학습용 정확도는 올라갔으나 평가용 정확도는 0.05이후 떨어진다
# 0.05 이후 부터 과적합이 나타나고 차이가 크게 벌어지는 0.3에서 뚜렷하게 나타난다.
# 적당한 학습률은 0.05이다
# 완전히 맞아 떨어지지 않는다, 아마도 파라미터를 조정했기 때문에 그런것같다.